# <center>ADT Lab 12 (10pts)</center>
## <center>Cassandra</center>

Setup Instructions-

- Go to the Astra DB website (https://astra.datastax.com/) and sign up for a free account.
- Click on database and hit Create Database
- Give database name

![first.png](attachment:first.png)

Create a keyspace named "lab 12"

![second.png](attachment:second.png)


![third.png](attachment:third.png)

- Go to the CQL Console and see if keyspace was created properly by running the command

`DESC KEYSPACES;`

![2-2.png](attachment:2-2.png)

You should see the keyspace that you created earlier. Keyspace is similar to a schema in relational dbms

#### Astra DB
Astra DB is a cloud-native database-as-a-service (DBaaS) offering provided by DataStax, which is designed to provide a fully managed and scalable NoSQL database service for modern applications. Astra DB is built on top of the Apache Cassandra database, but it includes additional features and capabilities that make it easier to use and more powerful.


- Now let’s use the keyspace to execute further queries, similar to what we do in RDBMS. First we select the schema to be used

`USE lab12;`

- Now lets create a people table using this query-

CREATE TABLE lab12.people (

    id uuid PRIMARY KEY,
    name text,
    height float,
    weight float
);

- Verify if the table is created by running

`DESC TABLES;`

Your output should be like-

![3.png](attachment:3.png)

In [1]:
!pip install cassandra-driver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 4.7 MB/s eta 0:00:00


In [2]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra.query import BatchStatement
from uuid import uuid4
import csv
import os

In [3]:
pip install --upgrade astrapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.5/333.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 25.6 MB/s eta 0:00:00


Go to the Database Tab and click on your active database. In the Database details panel, click on Download SCB(Secure Connect Bundle). Make sure that your current working directory contains the downloaded bundle
![fourth.png](attachment:fourth.png)

![fifth.png](attachment:fifth.png)

In [5]:
# Make sure your current working directory contains the downloaded bundle (secure-connect-shreya.zip)
os.listdir()

['.config', 'secure-connect-shimate.zip', 'sample_data']

 - Now go to the tokens tab generate and download the token as shown below. Select the Organization Administrator under roles. Mkae sure that your tokens file is also in the same working directory.
 ![sixth.png](attachment:sixth.png)

- Copy Paste your ClientID and ClientSecret below. Again, keep the token file in the same working directory


In [6]:
cloud_config= {
  'secure_connect_bundle': 'secure-connect-shimate.zip'
}
auth_provider = PlainTextAuthProvider('ZLpQCJytRlepNvYqoachJHKc', 'TvMwmF2OTPm83,J4ZeRUp2HKbQQZ_bQ8k+r7k9+1LQuZX,BjzW8RGJ2HXy5QG2PeiYoprtgbDL14Ad9Urb4oFZQd6I_yRLYfjNM004K823fpZ8NE2eHgd_ROnArTzTMU')
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()

In [7]:
# Read the data from the CSV file
with open('hw_200-1.csv', 'r') as f:
    reader = csv.reader(f)
    next(reader)
    data = [(uuid4(), row[0], float(row[1]), float(row[2])) for row in reader]

In [8]:
data[:10]

[(UUID('eac2a30e-4840-416b-bf87-89ebba92b363'), '1', 65.78, 112.99),
 (UUID('e520e7a4-47da-4557-bdb2-daeb9441fa99'), '2', 71.52, 136.49),
 (UUID('bac5e02f-7e91-4cde-ac0c-3410f74f196a'), '3', 69.4, 153.03),
 (UUID('086d1722-bd29-4f41-a594-dc235c8d9853'), '4', 68.22, 142.34),
 (UUID('dee452bd-1b42-4804-9d1f-4ce106c9178d'), '5', 67.79, 144.3),
 (UUID('a9d92fee-d9f8-44ba-a37a-8b36adfcde15'), '6', 68.7, 123.3),
 (UUID('3b666596-39eb-4f0c-bed0-1732888314e0'), '7', 69.8, 141.49),
 (UUID('1962c6f6-bbc0-4ebe-99a4-cd87b468d9a8'), '8', 70.01, 136.46),
 (UUID('3226d822-1720-4547-9226-3a11978907cc'), '9', 67.9, 112.37),
 (UUID('52fe34ee-9f16-457a-93e8-393c1b6517e1'), '10', 66.78, 120.67)]

In [9]:
# Set the keyspace
session.set_keyspace('lab12')

In [10]:
# Insert the data into the database
query = "INSERT INTO people (id, name, height, weight) VALUES (?, ?, ?, ?)"
prepared = session.prepare(query)
batch = BatchStatement()
for row in data:
    batch.add(prepared, row)
session.execute(batch)


- Now lets go to CQL console and see if values were inserted properly
- Go to CQL console; use lab12 keyspace and run the command-

`SELECT * FROM people;`

- You should see the following
![6.png](attachment:6.png)

- Now you can run some queries on the people's table

#### 1. Retrieve 10 records from the "people" table

In [11]:
rows = session.execute("SELECT * FROM people limit 10")
for row in rows:
    print(row.id, row.name, row.height, row.weight)

82585620-7bf2-4ae3-84a9-fb6f94a98645 22 68.87999725341797 143.5399932861328
c3c24be9-70de-48f3-b11f-e0e4e94537c1 157 70.9800033569336 158.9600067138672
e02ce9a3-6f69-45d7-98d3-75ea100395b1 199 68.0199966430664 127.47000122070312
ad989dd8-20ca-4ab2-a5e8-f1517fba0094 37 66.80000305175781 141.8000030517578
fecc1b75-78fa-40c9-9558-eb9e0674f522 62 69.0 137.5500030517578
4f34d71d-d557-4759-b914-cffdbdc3f673 32 65.80999755859375 120.75
65fdd32c-c635-4177-b894-07bb69f6230f 58 66.54000091552734 128.07000732421875
e8fff64a-b562-4aea-957e-ee0cab67cf4d 87 67.36000061035156 140.88999938964844
2bab5418-1162-4fd9-93fb-b0a41a8e6ecf 24 68.41999816894531 129.5
fbbdee39-31f7-4074-b1dc-a5ed0f321371 187 68.58000183105469 127.87000274658203


#### 2. Retrieve all data for individuals with a height greater than 70 inches

In [12]:
rows = session.execute("SELECT * FROM people WHERE height > 70")
for row in rows:
    print(row.id, row.name, row.height, row.weight)

InvalidRequest: Error from server: code=2200 [Invalid query] message="Cannot execute this query as it might involve data filtering and thus may have unpredictable performance. If you want to execute this query despite the performance unpredictability, use ALLOW FILTERING"

- You will get an error here. Now run the below cell

In [13]:
rows = session.execute("SELECT * FROM people WHERE height > 70 allow filtering")
for row in rows:
    print(row.id, row.name, row.height, row.weight)

c3c24be9-70de-48f3-b11f-e0e4e94537c1 157 70.9800033569336 158.9600067138672
a40a9af7-818c-45bf-b1fa-abd6f492b8ab 200 71.38999938964844 127.87999725341797
ce00e645-cd3c-4d6b-a2cf-338fa580449d 19 71.2300033569336 137.89999389648438
e7248a87-61a1-4e9d-8f3e-2dd066ee0284 72 70.83999633789062 134.02000427246094
ced142d2-4d64-42a3-b68d-075067daa4e5 96 70.55999755859375 131.8000030517578
7a1481f0-4d41-496d-8511-df63d84f4558 175 73.83000183105469 139.3000030517578
62957c2a-7e8a-45bf-a986-0f905cf8c16b 57 70.41000366210938 155.89999389648438
eac4526f-1bed-4349-b5a5-6992df9b7b3a 89 70.18000030517578 121.12000274658203
198f5373-8ede-4f50-a32f-5a258143186b 115 70.01000213623047 122.04000091552734
8f421f83-30f2-4649-bac1-7c3940888864 159 72.0199966430664 138.77999877929688
29309fbf-86d2-4dd7-8cb0-4339943d84a2 92 70.23999786376953 141.49000549316406
33423792-bc64-43ca-87b9-65ad6f63403f 27 70.83999633789062 142.4199981689453
1962c6f6-bbc0-4ebe-99a4-cd87b468d9a8 8 70.01000213623047 136.4600067138672
179

Why is the query running if we add  ALLOW FILTERING at the end?
- ALLOW FILTERING is a way to retrieve data from a table that doesn't involve using the primary key, but it can be an expensive operation as it requires scanning the entire table.
- In general, it's a best practice to avoid using ALLOW FILTERING whenever possible and to design your data model so that queries can be satisfied using the primary key. However, there may be cases where using ALLOW FILTERING is necessary or unavoidable.

3. Similarly now retrive data for individuals with a weight between 150 and 200 pounds

In [15]:
rows = session.execute("SELECT * FROM people WHERE weight >= 150 and weight < 200 allow filtering")
for row in rows:
    print(row.id, row.name, row.height, row.weight)

c3c24be9-70de-48f3-b11f-e0e4e94537c1 157 70.9800033569336 158.9600067138672
bac5e02f-7e91-4cde-ac0c-3410f74f196a 3 69.4000015258789 153.02999877929688
62957c2a-7e8a-45bf-a986-0f905cf8c16b 57 70.41000366210938 155.89999389648438
6d88cebd-a06b-42f7-bbd7-e45a2c390fc4 83 70.05000305175781 155.3800048828125
47e3edfe-497e-4712-8ed1-88da93b54dd2 139 73.9000015258789 151.38999938964844


Hint- Use allow filtering; Syntax will be similiar to normal SQL. We are actually using CQL here which is a subset of SQL

Expected output

![7.JPG](attachment:7.JPG)